# Live MLB Spring Training Games with Pitch-Level Data
Pull real-time spring training games and statcast pitch data for ongoing games.

In [11]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta

## 1. Get All Games and Show Available Types

In [12]:
def get_all_games(days_back=14, days_forward=14):
    '''Fetch all games for a dynamic date range.'''
    today = datetime.now().date()
    date_start = (today - timedelta(days=days_back)).strftime('%Y-%m-%d')
    date_end = (today + timedelta(days=days_forward)).strftime('%Y-%m-%d')
    
    print(f'Today: {today}')
    print(f'Searching for games from {date_start} to {date_end}\n')
    
    url = 'https://statsapi.mlb.com/api/v1/schedule'
    params = {'sportId': 1, 'startDate': date_start, 'endDate': date_end}
    try:
        response = requests.get(url, params=params, timeout=10)
        data = response.json()
        
        # API returns dict with 'dates' key containing list of day objects
        if isinstance(data, dict):
            all_games = []
            dates = data.get('dates', [])
            for date_obj in dates:
                if isinstance(date_obj, dict):
                    games = date_obj.get('games', [])
                    if isinstance(games, list):
                        all_games.extend(games)
            return all_games
        elif isinstance(data, list):
            return [g for g in data if isinstance(g, dict)]
        else:
            print(f'Unexpected response type: {type(data)}')
            return []
    except Exception as e:
        print(f'Error fetching games: {e}')
        return []

all_games = get_all_games(days_back=14, days_forward=14)
print(f'Found {len(all_games)} total games\n')

if all_games:
    print('All games by type:')
    game_types = {}
    for game in all_games:
        if not isinstance(game, dict):
            continue
        gt = game.get('gameType', 'Unknown')
        game_types[gt] = game_types.get(gt, 0) + 1
    
    print(f'Game types available: {game_types}\n')
    
    for game in all_games:
        if not isinstance(game, dict):
            continue
        away = game.get('teams', {}).get('away', {}).get('team', {}).get('name', 'N/A')
        home = game.get('teams', {}).get('home', {}).get('team', {}).get('name', 'N/A')
        status = game.get('status', {}).get('abstractGameState', 'N/A')
        gt = game.get('gameType', 'Unknown')
        date_time = game.get('gameDateTime', 'N/A')
        print(f'{date_time} | {away:20} @ {home:20} | Type: {gt:3} | Status: {status}')
else:
    print('No games found')

Today: 2026-03-23
Searching for games from 2026-03-09 to 2026-04-06

Found 378 total games

All games by type:
Game types available: {'S': 220, 'E': 6, 'R': 152}

N/A | Baltimore Orioles    @ St. Louis Cardinals  | Type: S   | Status: Final
N/A | St. Louis Cardinals  @ Houston Astros       | Type: S   | Status: Final
N/A | Tampa Bay Rays       @ Detroit Tigers       | Type: S   | Status: Final
N/A | Philadelphia Phillies @ Boston Red Sox       | Type: S   | Status: Final
N/A | Minnesota Twins      @ Atlanta Braves       | Type: S   | Status: Final
N/A | Cleveland Guardians  @ Kansas City Royals   | Type: S   | Status: Final
N/A | Athletics            @ Cincinnati Reds      | Type: S   | Status: Final
N/A | Los Angeles Angels   @ San Francisco Giants | Type: S   | Status: Final
N/A | Colorado Rockies     @ Chicago White Sox    | Type: S   | Status: Final
N/A | Seattle Mariners     @ Arizona Diamondbacks | Type: S   | Status: Final
N/A | Texas Rangers        @ San Diego Padres     | Type

## 2. Filter for Live Games (All Types)

In [13]:
# Filter for live/in progress games regardless of type
live_games = [g for g in all_games if isinstance(g, dict) and g.get('status', {}).get('abstractGameState') in ['Live', 'In Progress']]
print(f'\nLive/In Progress games: {len(live_games)}\n')

for game in live_games:
    if not isinstance(game, dict):
        continue
    away = game.get('teams', {}).get('away', {}).get('team', {}).get('name', 'N/A')
    home = game.get('teams', {}).get('home', {}).get('team', {}).get('name', 'N/A')
    status = game.get('status', {}).get('abstractGameState', 'N/A')
    gt = game.get('gameType', 'N/A')
    inning = game.get('liveData', {}).get('linescore', {}).get('currentInning', 'N/A')
    print(f'{away:20} @ {home:20} | Type: {gt} | Status: {status} | Inning {inning}')


Live/In Progress games: 6

Baltimore Orioles    @ Washington Nationals | Type: S | Status: Live | Inning N/A
Minnesota Twins      @ Boston Red Sox       | Type: S | Status: Live | Inning N/A
Atlanta Braves       @ Pittsburgh Pirates   | Type: S | Status: Live | Inning N/A
Chicago White Sox    @ Athletics            | Type: S | Status: Live | Inning N/A
New York Yankees     @ Chicago Cubs         | Type: S | Status: Live | Inning N/A
Seattle Mariners     @ San Diego Padres     | Type: S | Status: Live | Inning N/A


## 3. Extract Pitch Data Utilities

In [14]:
def safe_get(obj, *keys, default=None):
    '''Safely navigate nested dicts.'''
    current = obj
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        else:
            return default
    return current if current is not None else default

def get_game_livescore(game_pk):
    '''Get live play-by-play data.'''
    url = f'https://statsapi.mlb.com/api/v1/game/{game_pk}/livescore'
    try:
        response = requests.get(url, timeout=10)
        return response.json()
    except Exception as e:
        print(f'Error fetching livescore: {e}')
        return None

def extract_pitches_from_game(game_data):
    '''Extract all pitches from game data.'''
    pitches = []
    if not isinstance(game_data, dict):
        return pitches
    
    try:
        plays = game_data.get('plays', [])
        if not isinstance(plays, list):
            return pitches
        
        for play in plays:
            if not isinstance(play, dict):
                continue
            
            play_events = play.get('playEvents', [])
            if not isinstance(play_events, list):
                continue
            
            for event in play_events:
                if not isinstance(event, dict):
                    continue
                if event.get('type') != 'pitch':
                    continue
                
                details = safe_get(event, 'details', default={})
                pitch_info = {
                    'inning': safe_get(play, 'about', 'inning'),
                    'inningState': safe_get(play, 'about', 'halfInning'),
                    'batter_id': safe_get(play, 'matchup', 'batter', 'id'),
                    'pitcher_id': safe_get(play, 'matchup', 'pitcher', 'id'),
                    'pitch_type': safe_get(details, 'type', 'code'),
                    'velocity': safe_get(details, 'velocity'),
                    'spin_rate': safe_get(details, 'spinRate'),
                    'break_x': safe_get(details, 'breaks', 'breakX'),
                    'break_z': safe_get(details, 'breaks', 'breakZ'),
                    'px': safe_get(details, 'coordinates', 'pX'),
                    'pz': safe_get(details, 'coordinates', 'pZ'),
                    'description': safe_get(event, 'description', default='')
                }
                pitches.append(pitch_info)
    except Exception as e:
        print(f'Error extracting pitches: {e}')
    
    return pitches

## 4. Fetch Pitch Data from Live Games

In [15]:
all_pitches = []

for game in live_games:
    try:
        game_pk = game['gamePk']
        away = safe_get(game, 'teams', 'away', 'team', 'name', default='Away')
        home = safe_get(game, 'teams', 'home', 'team', 'name', default='Home')
        status = safe_get(game, 'status', 'abstractGameState', default='Unknown')
        
        print(f'Fetching: {away} @ {home} ({status})...')
        game_data = get_game_livescore(game_pk)
        
        if game_data:
            pitches = extract_pitches_from_game(game_data)
            print(f'  Found {len(pitches)} pitches')
            
            for pitch in pitches:
                pitch['game_pk'] = game_pk
                pitch['away_team'] = away
                pitch['home_team'] = home
            
            all_pitches.extend(pitches)
    except Exception as e:
        print(f'  Error processing game: {e}')
    
    print()

print(f'Total pitches: {len(all_pitches)}')

Fetching: Baltimore Orioles @ Washington Nationals (Live)...
  Found 0 pitches

Fetching: Minnesota Twins @ Boston Red Sox (Live)...
  Found 0 pitches

Fetching: Atlanta Braves @ Pittsburgh Pirates (Live)...
  Found 0 pitches

Fetching: Chicago White Sox @ Athletics (Live)...
  Found 0 pitches

Fetching: New York Yankees @ Chicago Cubs (Live)...
  Found 0 pitches

Fetching: Seattle Mariners @ San Diego Padres (Live)...
  Found 0 pitches

Total pitches: 0


## 5. DataFrame Analysis

In [16]:
if all_pitches:
    df = pd.DataFrame(all_pitches)
    print(f'Shape: {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    print()
    print(df.head(10))
else:
    print('No pitch data available. Check if any games are live above.')

No pitch data available. Check if any games are live above.


## 6. Real-Time Monitoring (Optional)

In [ ]:
def monitor_live_games(interval_seconds=30, max_duration_minutes=60):
    '''Continuously monitor live games.'''
    start_time = time.time()
    max_seconds = max_duration_minutes * 60
    all_data = []
    last_counts = {}
    
    while (time.time() - start_time) < max_seconds:
        try:
            games = get_all_games()
            live = [g for g in games if isinstance(g, dict) and g.get('status', {}).get('abstractGameState') in ['Live', 'In Progress']]
            
            if not live:
                print(f'[{datetime.now().strftime("%H:%M:%S")}] No live games')
                time.sleep(interval_seconds)
                continue
            
            print(f'[{datetime.now().strftime("%H:%M:%S")}] {len(live)} live game(s)')
            
            for game in live:
                pk = game['gamePk']
                away = safe_get(game, 'teams', 'away', 'team', 'name', default='Away')
                home = safe_get(game, 'teams', 'home', 'team', 'name', default='Home')
                
                data = get_game_livescore(pk)
                if data:
                    pitches = extract_pitches_from_game(data)
                    curr = len(pitches)
                    prev = last_counts.get(pk, 0)
                    
                    if curr > prev:
                        print(f'  {away} @ {home}: +{curr - prev} pitch(es)')
                        last_counts[pk] = curr
                        
                        for p in pitches:
                            p['game_pk'] = pk
                            p['away_team'] = away
                            p['home_team'] = home
                            p['timestamp'] = datetime.now().isoformat()
                        
                        all_data.extend(pitches)
            
            time.sleep(interval_seconds)
        except Exception as e:
            print(f'Error: {e}')
            time.sleep(interval_seconds)
    
    return all_data

# Uncomment to run:
data = monitor_live_games(interval_seconds=30, max_duration_minutes=60)
df_live = pd.DataFrame(data)
print(f'Collected {len(df_live)} pitches')

SyntaxError: invalid syntax (609841821.py, line 50)

In [23]:
import statsapi

# 1. Get today's games to find a live game_pk
today = statsapi.schedule()
for game in today:
    #print(f"Game ID: {game['game_id']} | {game['away_name']} @ {game['home_name']} | Status: {game['status']}")
    if game['status'] in ['Live', 'In Progress']:
        print(f"  Found live game: {game['game_id']} - {game['away_name']} @ {game['home_name']}")

# 2. Pull live data for a specific game ID
game_id = 747175  # Example Game ID
live_data = statsapi.get('game', {'gamePk': 831735})

# print(live_data)
# pd.DataFrame(live_data['liveData']['plays']['plays'][0]['playEvents'])
# This returns a massive JSON object containing the current at-bat, 
# the current pitch count, and real-time play-by-play!

  Found live game: 832036 - Minnesota Twins @ Boston Red Sox
  Found live game: 831447 - Atlanta Braves @ Pittsburgh Pirates
  Found live game: 831915 - Chicago White Sox @ Athletics
  Found live game: 831780 - New York Yankees @ Chicago Cubs
  Found live game: 831735 - Seattle Mariners @ San Diego Padres
